# DA Evaluation — Exploration Tools (companion to notebook 07)

Interactive / exploratory views moved out of the paper notebook `07_Data_Assimilation_Embedding_Parquet_Visualizations.ipynb` 
so that 07 only holds paper and supplement outputs. Run the setup cells below first: they are copies of 07's 
setup, experiment/`SELECTION` and settings cells. **Keep them in sync with 07** (same `ACTIVE_RUN_ID` and `SELECTION`), 
so reference models match the paper.

Contents: top-N leaderboard, optimal-basin distribution, ANOVA (optional), interactive risk–return, stratified CDFs, 
linked diagnostic dashboard.


In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

# Add repository root and tutorial utils to sys.path
cwd = Path(os.getcwd()).resolve()
repo_root = next(
    (p for p in [cwd, cwd.parent, cwd.parent.parent, cwd.parent.parent.parent, Path('/usr/local/google/home/kruparell/openhydronets_next')] if (p / 'tutorial' / 'utils').is_dir()),
    cwd
)
tut_utils = repo_root / 'tutorial' / 'utils'
if str(tut_utils) not in sys.path:
    sys.path.insert(0, str(tut_utils))

from da_eval import DAAnalysisSession
print("DA Evaluation environment loaded successfully.")

## Stage 1: Data Ingestion & Benchmark Initialization

In this section, we initialize the evaluation session. The engine automatically:
1. Scopes data loading strictly to this experiment's directory (preventing cross-contamination).
2. **Optional In-Distribution / Overlap Filtering**: Supports filtering evaluation catchments to just the overlapping training/pretrained basins (`filter_basins=...`), filtering out unseen out-of-distribution global catchments.
3. Detects available forecast horizons (including dynamic **$t+0$ nowcast** detection).
4. Synthesizes the **Per-Basin Best DA** oracle upper bound.
5. Extracts physical morphology, slope, elevation, snow fraction, and aridity from `Caravans_V2/attributes.zarr`.

In [ ]:
# Curated Experiment Selection Registry
EXPERIMENTS = {
    # 0000. Stage 53: 4,287-Basin 6-Year (2017-01-01 .. 2023-09-30) 84-Config S / S+H / S+H+F Factorial Suite [XID 294056488]
    #       filtered_basins_nse_gt_0.5.txt (all 4,287 catchments), CMAL mixture_mean, NSE loss,
    #       bg=0.5, tol=0.05, epochs=50, 4 shards/config (85 configs x 4 shards = 340 V100 Borg jobs).
    #       - S (Static-only, 12 cfgs):      w in {7, 14, 30, 90} x lr in {0.001, 0.005, 0.1}
    #       - S+H (Hindcast+Static, 36 cfgs): w in {7, 14, 30, 90} x lr_dyn in {0.001, 0.005, 0.1} x r in {0.1, 0.3, 1.0}
    #       - S+H+F (Triple, 36 cfgs):       w in {7, 14, 30, 90} x lr_dyn in {0.001, 0.005, 0.1} x r in {0.1, 0.3, 1.0}
    #       PP benchmark rows are generated offline from the staged Baseline zarr (da_eval.ar1_metrics).
    "294056488": {
        "name": "da_st53_4287b_6yr_s_sh_shf_84cfg_20260928",
        "mode": "embedding",
        "cns_dir": "/cns/jn-d/home/floods/hydro_model/work/kruparell/large_scale_param_selection_results/da_st53_4287b_6yr_s_sh_shf_84cfg_20260928",
        "local_dir": "/usr/local/google/home/kruparell/openhydronets_next/tutorial/data/eval_staging/da_st53_4287b_6yr_s_sh_shf_84cfg_20260928",
    },
    # 000. Stage 52: 1,000-Basin 1-Year (2017) 84-Config S / S+H / S+H+F Factorial Suite [XID 293997545]
    #      stratified_1000_screen.txt basins (n=672 finite across all 7 leads), CMAL mixture_mean, NSE loss,
    #      bg=0.5, tol=0.05, epochs=50, 1 shard/config (85 V100 jobs = 1 baseline + 84 DA configs).
    #      - S (Static-only, 12 cfgs):      w in {7, 14, 30, 90} x lr in {0.001, 0.005, 0.1}
    #      - S+H (Hindcast+Static, 36 cfgs): w in {7, 14, 30, 90} x lr_dyn in {0.001, 0.005, 0.1} x r in {0.1, 0.3, 1.0}
    #      - S+H+F (Triple, 36 cfgs):       w in {7, 14, 30, 90} x lr_dyn in {0.001, 0.005, 0.1} x r in {0.1, 0.3, 1.0}
    #      PP benchmark rows are generated offline from the staged Baseline zarr (da_eval.ar1_metrics).
    "293997545": {
        "name": "da_st52_1000b_1yr_s_sh_shf_84cfg_20260928",
        "mode": "embedding",
        "cns_dir": "/cns/jn-d/home/floods/hydro_model/work/kruparell/large_scale_param_selection_results/da_st52_1000b_1yr_s_sh_shf_84cfg_20260928",
        "local_dir": "/usr/local/google/home/kruparell/openhydronets_next/tutorial/data/eval_staging/da_st52_1000b_1yr_s_sh_shf_84cfg_20260928",
    },
    # 00. Stage 50: 4,287-Basin 6-Year (2017-01-01 .. 2023-09-30) Winners Benchmark [XID 293022110]
    #     11 global DA winners (4 Hindcast+Static, 2 Dual, 2 Triple, 2 Hindcast-Only, 1 Static-Only)
    #     + E_baseline_0da across all 4,287 filtered catchments (filtered_basins_nse_gt_0.5.txt),
    #     CMAL mixture_mean point estimates, 12 configs x 8 shards = 96 V100 Borg jobs.
    #     PP benchmark rows are generated offline from the staged Baseline zarr (da_eval.ar1_metrics).
    "293022110": {
        "name": "da_st50_4287b_6yr_winners_20260924",
        "mode": "embedding",
        "cns_dir": "/cns/jn-d/home/floods/hydro_model/work/kruparell/large_scale_param_selection_results/da_st50_4287b_6yr_winners_20260924",
        "local_dir": "/usr/local/google/home/kruparell/openhydronets_next/tutorial/data/eval_staging/da_st50_4287b_6yr_winners_20260924",
    },
    # 01. Stage 51: 500-Basin 2-Year (2017-2018) 72-Config Hindcast(+Static) LR-Ratio / Tol / BG Sweep [XID 293040723]
    #     stratified_500_screen.txt basins, CMAL mixture_mean, NSE loss, 1 shard/config (73 V100 jobs).
    #     Config IDs: S51<group>_w<w>_lrd<lr_dyn>_r<lr_stat/lr_dyn>_bg<bg>_tol<tol>_ep<epochs>
    #     (r0 = hindcast-only; r>0 = hindcast + static with lr_stat = r * lr_dyn).
    #     - S51A (29): ratio sweep r in {0, 0.1, 0.3, 1}, w in {7, 14, 30, 90}, lr_dyn in {0.005, 0.01}, bg 0.5
    #     - S51B (12): heavy regularization bg in {1, 2, 5} at lr_dyn 0.1, r 0.1, w in {7, 14, 30, 90}
    #     - S51C (22): early-stopping tol in {0.01, 0.05, 0.1} x epochs in {25, 50}
    #     - S51E (5):  extreme LR / bg probes at w in {30, 90};  S51F (4): w = 365 long-window probes
    #     PP benchmark rows are generated offline from the staged Baseline zarr (da_eval.ar1_metrics).
    "293040723": {
        "name": "da_st51_500b_l47_hs_ratio_tol_bg_20260925",
        "mode": "embedding",
        "cns_dir": "/cns/jn-d/home/floods/hydro_model/work/kruparell/large_scale_param_selection_results/da_st51_500b_l47_hs_ratio_tol_bg_20260925",
        "local_dir": "/usr/local/google/home/kruparell/openhydronets_next/tutorial/data/eval_staging/da_st51_500b_l47_hs_ratio_tol_bg_20260925",
    },
    # 0. Stage 49: 4,287-Basin 2-Year (2017-2018) Full 20-Config Global Winners Benchmark [XID 292697766]
    #    Evaluates 19 global DA winners across all 5 categories (Hindcast+Static, Triple, Dual,
    #    Hindcast-Only, Static-Only) + E_baseline_0da evaluated across all 4,287 global catchments
    #    under CMAL mixture_mean point estimates (100 V100 Borg jobs).
    "292697766": {
        "name": "da_st49_4287b_winners_suite_2yr_20260924",
        "mode": "embedding",
        "cns_dir": "/cns/jn-d/home/floods/hydro_model/work/kruparell/large_scale_param_selection_results/da_st49_4287b_winners_suite_2yr_20260924",
        "local_dir": "/usr/local/google/home/kruparell/openhydronets_next/tutorial/data/eval_staging/da_st49_4287b_winners_suite_2yr_20260924",
    },
    # 0a. Stage 48: 500-Basin 2-Year (2017-2018) 72-Config Tier 1 Coupled Hindcast+Static Sweep [XID 292693312]
    #    Evaluates 72 hindcast_static coupled DA configs (loss_w = w in {7, 14, 30, 90},
    #    lr in {0.1, 0.01, 0.001}, bg in {0.0, 0.05, 0.5}, tol in {None, 0.05}, epochs=50)
    #    + E_baseline_0da evaluated under CMAL mixture_mean point estimates.
    "292693312": {
        "name": "da_st48_500b_hs_coupled_mixture_mean_20260924",
        "mode": "embedding",
        "cns_dir": "/cns/jn-d/home/floods/hydro_model/work/kruparell/large_scale_param_selection_results/da_st48_500b_hs_coupled_mixture_mean_20260924",
        "local_dir": "/usr/local/google/home/kruparell/openhydronets_next/tutorial/data/eval_staging/da_st48_500b_hs_coupled_mixture_mean_20260924",
    },
    # 0b. Stage 45: 2,000-Basin 2-Year (2017-2018) 10-Config Winners Benchmark [XID 292499199]
    #    Evaluates 9 top Embedding DA winners (Hindcast + Static W2B/S45_HS and Hindcast-Only W2/S45_H
    #    across w in {14, 30, 60}, loss in {NSE, MSE}, lw in {5, 10, 60}) + E_baseline_0da across
    #    2,000 stratified basins (10 shards/cfg = 100 V100 Borg jobs).
    "292499199": {
        "name": "da_embeddings_st45_2000b_2yr_winners_20260923",
        "mode": "embedding",
        "cns_dir": "/cns/jn-d/home/floods/hydro_model/work/kruparell/large_scale_param_selection_results/da_embeddings_st45_2000b_2yr_winners_20260923",
        "local_dir": "/usr/local/google/home/kruparell/openhydronets_next/tutorial/data/eval_staging/da_embeddings_st45_2000b_2yr_winners_20260923",
    },

}

# Select active run XID:
# - '294056488': Stage 53 4,287-Basin 6-Year (2017-01 .. 2023-09) 84-Config S / S+H / S+H+F Factorial Suite (mixture_mean)
# - '293997545': Stage 52 1,000-Basin 1-Year (2017) 84-Config S / S+H / S+H+F Factorial Suite (mixture_mean)
# - '293022110': Stage 50 4,287-Basin 6-Year (2017-01 .. 2023-09) 12-Config Winners Benchmark (mixture_mean)
# - '293040723': Stage 51 500-Basin 2-Year (2017-2018) 72-Config LR-Ratio / Tol / BG Sweep (mixture_mean)
# - '292697766': Stage 49 4,287-Basin 2-Year (2017-2018) Full 20-Config Global Winners Benchmark (mixture_mean)
# - '292693312': Stage 48 500-Basin 2-Year (2017-2018) 72-Config Coupled Hindcast+Static Sweep (mixture_mean)
# - '292499199': Stage 45 2,000-Basin 2-Year (2017-2018) 10-Config Winners Benchmark (CSV + Zarr staged)

ACTIVE_RUN_ID = "293022110"

selected_exp = EXPERIMENTS[ACTIVE_RUN_ID]
DATA_DIR = selected_exp["local_dir"]
CNS_DIR = selected_exp["cns_dir"]
MODE = selected_exp["mode"]
ATTR_ZARR = "/usr/local/google/home/kruparell/Caravans_V2/attributes.zarr"

# Sync Toggle: Set True to pull new shards from CNS; False for ultra-fast local loading once staged.
SYNC_FROM_CNS = False

# Basin filtering: Filter strictly to overlapping pretrained/training catchments
FILTER_BASINS_PATH = "/usr/local/google/home/kruparell/openhydronets_next/example-configs/filtered_basins_nse_gt_0.5.txt"
FILTER_TO_OVERLAPPING = False  # Set to False to analyze all evaluated catchments

# Report how many configs are actually staged locally (auto-syncs from CNS if 0 staged locally).
from pathlib import Path as _Path
_staged_csvs = list(_Path(DATA_DIR).glob("**/test_metrics*.csv"))
_staged_zarrs = list(_Path(DATA_DIR).glob("**/test_results*.zarr"))
_staged_parquets = list(_Path(DATA_DIR).glob("*.parquet"))
print(f"Loading Experiment XID {ACTIVE_RUN_ID} ({selected_exp['name']}) [Mode: {MODE}]...")
print(f"Locally staged files: {len(_staged_csvs)} metric CSVs, {len(_staged_zarrs)} Zarr stores, {len(_staged_parquets)} parquets")

# PP benchmark rows (fixed rho + per-basin best rho) are synthesised from the staged Baseline zarr the
# first time an experiment is loaded (~20 s per 1k basins-years; skipped when the CSVs already exist).
if _staged_zarrs:
    from da_eval.ar1_metrics import build_ar1_metrics
    build_ar1_metrics(DATA_DIR)

SELECTION = dict(metric="NSE Skill Score", leads=(4,5,6,7), agg="mean", global_stat="median")

session = DAAnalysisSession(
    selection=SELECTION,
    mode=MODE,
    data_dir=DATA_DIR,
    attr_zarr_path=ATTR_ZARR,
    cns_dir=CNS_DIR if (SYNC_FROM_CNS or (not _staged_csvs and not _staged_parquets)) else None,
    filter_basins=FILTER_BASINS_PATH if FILTER_TO_OVERLAPPING else None,
    load_timeseries=False,  # Fast loading for full 4,287-basin metrics (~8s)
)

### Reference models (set once for the whole notebook)

`SELECTION` in the cell above defines **Global-Best DA**, **Per-Basin DA**, **Global-Best PP** and **Per-Basin PP** for every later stage. No plot or table chooses its own reference models, and the same Global-Best config is shown at every lead time. Plot titles and legends state the rule (e.g. *selected on median NSE skill score, t+1*).

> [!NOTE]
> Selection is **in-sample**: models are chosen on the same period they are evaluated on. Per-Basin models are therefore oracle upper bounds, and the Global-Best is optimistic by a (small) selection bias.


In [ ]:
# Display units: NSE skill score shown as a percentage in every plot / table (SS_NSE x 100 = % reduction in
# squared error vs the open-loop). Presentation only - thresholds, colour limits and arguments stay fractions
# (e.g. vmin=-0.2 -> -20%, bin_thresholds "0.01, 0.05" -> 1% / 5%). Set False for raw fractions.
session.set_ss_percent(True)

# Colours / line styles of the reference models (Baseline, Global / Per-Basin DA, Global / Per-Basin PP),
# used by every plot below. "cvd_safe" = colour-blind safe (default); "legacy" = the earlier Google colours.
session.set_palette("cvd_safe")

# Ranking behind the selection (DA and PP pools, winner = bold) and each config's share of Per-Basin wins.
session.selection_summary()


In [ ]:
# Top-N Configuration Leaderboard across all forecast horizons (increase top_n=80 to view full grid)
session.show_top_n_table(top_n=15)

## Stage 3: Optimal Configuration Allocation (Globally Best vs. Per-Basin DA)

* **Panel A (Horizontal Bar Chart)**: Top $N$ configurations ranked by the number of catchments ($N_{\text{basins}}$) where they achieve optimal performance, highlighting the **Globally Best DA Configuration** ($\star$).
* **Panel B (Market Share Donut Chart)**: Visualizes catchment win share ($\%$) across top models vs. the remaining sweep configurations.
* **Optimal Allocation Matrix**: Summary table reporting per-basin win counts, win share ($\%$) and global median $\Delta\text{NSE}$.

In [ ]:
# 1. Top-N Configurations by Optimal Basin Count (Bar Chart + Market Share Donut)
# Winners = the Per-Basin DA picks of the notebook-wide SELECTION; lead_time only sets the displayed medians.
session.plot_optimal_basin_distribution(top_n=10, lead_time=1, plot_type="combined")

# 2. Optimal Basin Allocation Summary Table
session.show_optimal_basin_distribution_table(top_n=10, lead_time=1)

In [ ]:
# Optional: Classical ANOVA Rank Contribution (%) & Marginal Main Effects Curves
# session.plot_hyperparameter_effects(lead_time=1)

### Stage 4b: Risk–Return (Pareto) View of DA Configurations

One point per DA configuration. For each basin, the score is the chosen metric averaged (or summed) over **any set of lead times** you toggle, e.g. t+7 only, t+4–7, or t+1,3,7. Only basins where every selected lead is finite for every config are used. Across those basins:

* **Y (return)**: typical gain (median by default; mean, Q25 or % improved also available).
* **X (risk)**: downside (10th percentile by default; Q05, CVaR over the worst 10%, or % harmed also available). Right/up is better.
* **Colour / marker / label by** any hyperparameter that varies in the run. **Hold fixed** picks a controlled slice; everything else stays as the grey cloud.
* The red dashed line is the **Pareto frontier** (non-dominated configs), listed in the table below the plot. Reference points show Baseline, Global-Best DA, Per-Basin DA and the PP post-processors.
* **Basins** restricts the plot to one Köppen-Geiger climate group.

In [ ]:
# Interactive risk–return suite. Everything is also available as arguments for static paper figures, e.g.:
#   session.plot_risk_return(leads=(4, 5, 6, 7), metric="delta", agg="sum", risk_stat="q10", return_stat="median",
#                            color_by="Learning Rate", marker_by="Window (Days)",
#                            fixed={"LR Ratio": 0.1, "BG Weight": 0.5, "Tolerance": "0.05", "Epochs": 50})
# which returns the Pareto-frontier table. session.get_risk_return_table(...) returns (per-config table, info).
session.plot_risk_return(
    leads=(4, 5, 6, 7),
    metric="skill",          # "skill" | "delta" | "nse"
    agg="mean",              # "mean" | "sum" over the selected leads
    risk_stat="q10",         # median | mean | q05 | q10 | q25 | cvar10 | pct_improved | pct_harm
    return_stat="median",
    color_by="Learning Rate",
    marker_by="Window (Days)",
    interactive=True,
)

### Stage 5b: Stratified Catchment CDFs & Basin-Type Leaderboards (Morphological Filtering)

Test the hypothesis that **different configurations are optimal for different catchment types** (e.g. Arid vs Humid, Poor vs Strong Baseline, Low vs High Precipitation):
* **Live Interactive Attribute Filtering**: Filter catchments by:
  * **Baseline Quality Tier**: `Base NSE` range (e.g. Low `< 0.50`, Mid `0.50–0.70`, High `≥ 0.70`).
  * **Aridity Regimes**: UNEP Aridity Index ($P / \text{PET}$) range (e.g. Arid `< 0.50`, Sub-Humid `0.50–0.80`, Humid `≥ 0.80`).
  * **Precipitation Regimes**: Mean annual precipitation range (e.g. Low `< 1.5`, Moderate `1.5–3.5`, High `> 3.5` mm/day).
  * **Drainage Area & Snow Fraction**: Area in km² or cryosphere snow-melt fraction.
* **Stratified Sub-Cohort Leaderboard Scorecard**: Automatically ranks all sweep configurations on the filtered catchment cohort, highlighting the **winning configuration** (🥇) for that specific basin type along with win rates and degradation rates.
* **Dual-Panel Stratified ECDF & Decay Curve**:
  * **Panel A**: Empirical CDF comparing the candidate configurations specifically on the filtered catchments.
  * **Panel B**: Lead time memory persistence decay trajectory ($t+1$ to $t+7$) for the top configs on this stratum.

In [ ]:
# Launch Interactive Stratified Catchment ECDF & Leaderboard Suite
# Adjust the Preset dropdown or drag the range sliders to test different basin types in real time.
# Supported metrics: "skill_nse", "delta_nse", "delta_kge", "NSE", "KGE"
session.plot_stratified_cdfs(lead_time=1, metric="skill_nse", interactive=True)

## Stage 8: Linked Interactive Diagnostic Dashboard (Clickable Map)

An interactive application enabling spatial and catchment-level inspection:
* **5-Category Discrete Binned Map Scale (with On-the-Fly Thresholds)**: Defaults to 5 symmetric categories around zero (`< -0.05`, `-0.05 to -0.01`, `-0.01 to +0.01`, `+0.01 to +0.05`, `> +0.05`) with per-bin gauge percentages on the colorbar. Edit `Bin Thresholds (±)` on the fly (e.g., `"0.01, 0.05"` or `"0.02, 0.10"`) or switch `Scale` between **5-Bin Discrete** and **Continuous** (`[-0.2, +0.2]`).
* **Circle Size & Continental Zoom Presets**: Adjust **`Circle Size`** (`marker_size`, default `10`) with magnitude-sorted Z-ordering (so neutral `±0.01` gauges never obscure strong gains/losses) and zoom directly into **Global**, **North America**, **South America**, **Europe**, or **Australia**.
* **Complete Streamflow Period Filter**: Toggle **`No Missing Streamflow (q_obs) in Period`** (or pass `filter_complete_streamflow=True`) to restrict the map and basin selector to gauges with 100% complete observed discharge during the active plotting year.
* **Linked Hydrograph & Horizon Persistence**: Includes the active `Basin ID` directly in the hydrograph and persistence titles, colors persistence bars by the 5-bin palette, and highlights the active lead-time horizon (`t+1`..`t+7`).

In [ ]:
# Launch Unified Spatial Catchment Map & Linked Diagnostic Dashboard
# All options below are also adjustable interactively in the dashboard toolbar:
#   - color_by: "skill_nse" (NSE Skill Score, default), "delta" (ΔNSE), "da", or "baseline"
#   - color_mode: "binned" (5-category discrete scale) or "continuous"
#   - bin_thresholds: "0.01, 0.05" -> bins (< -0.05, -0.05..-0.01, ±0.01, +0.01..+0.05, > +0.05)
#   - marker_size: Circle size on map (default 10.0)
#   - region: "global", "north_america", "south_america", "europe", or "australia"
#   - vmin, vmax: Continuous colorbar bounds (default -0.2, 0.2)
#   - filter_complete_streamflow: If True, only show basins with zero missing q_obs in `year`
#   - koppen: Köppen-Geiger underlay (Beck et al. 2023, 1991–2020): "off", "groups" (A–E) or "classes" (30)
#   - koppen_filter: "all", a group letter ("B" = Arid) or classes ("BSk, Cfb") — restricts plotted catchments
#   - koppen_alpha: underlay opacity
session.launch_dashboard(
    year=2017,
    color_by="skill_nse",
    color_mode="binned",
    bin_thresholds="0.01, 0.05",
    marker_size=10.0,
    region="global",
    vmin=-0.2,
    vmax=0.2,
    filter_complete_streamflow=False,
    koppen="groups",
    koppen_filter="all",
    koppen_alpha=0.4,
)